# DuckDB — praktyczny przewodnik: od podstaw po zaawansowane użycie

DuckDB to **in-process, kolumnowy silnik OLAP** — działa wbudowany w proces
Pythona (jak SQLite), ale zamiast pod obciążenie transakcyjne (OLTP) jest
zoptymalizowany pod analitykę: przetwarzanie kolumnowe, wektoryzowana
egzekucja zapytań, równoległość na wątkach, i — co najważniejsze w Twoim
stacku — **zero-copy odpytywanie DataFrame'ów Pandas/Polars i plików
Parquet/CSV bezpośrednio przez SQL**, bez osobnego serwera bazodanowego.

**Instalacja:**
```
pip install duckdb
```

Notebook zakłada znajomość SQL (SQL Server) i pracy z Pandas/Polars —
skupia się na tym, co w DuckDB jest **inne** albo **dodatkowe** względem
tego, co już znasz.

In [1]:
import duckdb
import pandas as pd
import polars as pl
import numpy as np
from pathlib import Path

print(duckdb.__version__)

1.5.5


## 1. Dwa sposoby pracy: `duckdb.sql()` vs `duckdb.connect()`

- **`duckdb.sql(...)`** — najszybsza droga do ad-hoc zapytania. Korzysta z
  domyślnego, globalnego połączenia in-memory (`duckdb.default_connection`)
  współdzielonego w całym procesie. Wygodne w notebooku/skrypcie
  jednorazowym.
- **`duckdb.connect(...)`** — jawne, niezależne połączenie. Potrzebne, gdy:
  - chcesz **persystencję** — połączenie z plikiem `.duckdb` zamiast pamięci
    (dane przetrwają między uruchomieniami, jak w SQL Server, tylko lokalnie
    w jednym pliku),
  - potrzebujesz **izolacji** — np. kilku niezależnych sesji w tym samym
    procesie, każda z własnym stanem (utworzone tabele, ustawienia),
  - piszesz kod produkcyjny, gdzie jawne zarządzanie cyklem życia
    połączenia (`con.close()`) jest lepszą praktyką niż poleganie na
    globalnym stanie modułu.

`duckdb.connect()` bez argumentu (albo z `":memory:"`) też tworzy bazę
**w pamięci** — różnicą względem `duckdb.sql()` jest tylko to, że dostajesz
osobne, nazwane połączenie zamiast korzystania z globalnego.

In [2]:
# Ad-hoc, na domyślnym połączeniu in-memory
duckdb.sql("SELECT 42 AS odpowiedz, current_date AS dzis").show()

┌───────────┬────────────┐
│ odpowiedz │    dzis    │
│   int32   │    date    │
├───────────┼────────────┤
│        42 │ 2026-09-23 │
└───────────┴────────────┘



In [3]:
# Połączenie plikowe — dane przetrwają zamknięcie procesu
db_path = Path("analityka_demo.duckdb")
con = duckdb.connect(str(db_path))

con.sql("CREATE OR REPLACE TABLE test AS SELECT 1 AS x, 'a' AS y")
con.sql("SELECT * FROM test").show()

con.close()
print(f"Plik bazy na dysku: {db_path.exists()}, rozmiar: {db_path.stat().st_size} B")

┌───────┬─────────┐
│   x   │    y    │
│ int32 │ varchar │
├───────┼─────────┤
│     1 │ a       │
└───────┴─────────┘

Plik bazy na dysku: True, rozmiar: 536576 B


## 2. Relacja (`DuckDBPyRelation`) i leniwa ewaluacja

`duckdb.sql(...)` (i `con.sql(...)`) **nie wykonuje** zapytania od razu —
zwraca obiekt `DuckDBPyRelation`, czyli leniwy, niewykonany jeszcze plan.
Zapytanie faktycznie liczy się dopiero przy materializacji:

| Metoda | Wynik |
|---|---|
| `.df()` | `pandas.DataFrame` |
| `.pl()` | `polars.DataFrame` |
| `.arrow()` | `pyarrow.Table` |
| `.fetchall()` / `.fetchnumpy()` | lista tuple / dict NumPy — jak w DB-API |
| `.show()` | tylko podgląd tekstowy, nic nie zwraca |

Relację można też dalej **komponować** (`.filter()`, `.project()`,
`.limit()`, `.order()`) — DuckDB dokłada te operacje do planu i wciąż nic
nie liczy, dopóki nie wywołasz materializacji. To odpowiednik lazy frame
z Polars (`pl.LazyFrame`) — tylko że tu "frame" to relacja SQL.

In [4]:
rel = duckdb.sql("SELECT * FROM range(1, 1000) AS t(n)")   # nic jeszcze nie policzone
rel = rel.filter("n % 7 = 0").project("n, n * n AS n_kwadrat").limit(5)

print(type(rel))
rel.df()   # dopiero tu DuckDB faktycznie wykonuje plan

<class '_duckdb.DuckDBPyRelation'>


,n,n_kwadrat
0,7,49
1,14,196
2,21,441
3,28,784
4,35,1225


## 3. Odpytywanie DataFrame'ów bezpośrednio — zero-copy (Pandas i Polars)

DuckDB widzi zmienne Pythona z bieżącego scope po nazwie — bez importu,
rejestracji czy kopiowania danych (tzw. *replacement scan*). Dla Pandas i
Polars odbywa się to przez Arrow (zero-copy albo prawie zero-copy), więc
koszt jest znikomy nawet przy większych ramkach.

In [5]:
rng = np.random.default_rng(42)
n = 200_000

sprzedaz = pd.DataFrame({
    "id_transakcji": np.arange(1, n + 1),
    "produkt": rng.choice(["A", "B", "C", "D"], size=n),
    "miasto": rng.choice(["Warszawa", "Krakow", "Poznan", "Gdansk"], size=n),
    "ilosc": rng.integers(1, 10, size=n),
    "cena": rng.uniform(10, 500, size=n).round(2),
    "data": pd.date_range("2025-01-01", periods=n, freq="min"),
})
sprzedaz.head(3)

,id_transakcji,produkt,miasto,ilosc,cena,data
0,1,A,Gdansk,7,244.87,2025-01-01 00:00:00
1,2,D,Gdansk,4,211.59,2025-01-01 00:01:00
2,3,C,Gdansk,9,385.66,2025-01-01 00:02:00


In [6]:
# DuckDB widzi `sprzedaz` (Pandas) po prostu po nazwie w zapytaniu
duckdb.sql("""
    SELECT produkt,
           COUNT(*)                    AS n_transakcji,
           ROUND(SUM(ilosc * cena), 2) AS przychod
    FROM sprzedaz
    GROUP BY produkt
    ORDER BY przychod DESC
""").df()

,produkt,n_transakcji,przychod
0,C,50323,63787584.55
1,B,49948,63688960.08
2,D,49922,63616189.38
3,A,49807,63036155.86


In [7]:
# Dokładnie tak samo działa to z Polars — żadnej dodatkowej konwersji
sprzedaz_pl = pl.from_pandas(sprzedaz)

duckdb.sql("""
    SELECT miasto, COUNT(DISTINCT produkt) AS liczba_produktow
    FROM sprzedaz_pl
    GROUP BY miasto
    ORDER BY miasto
""").pl()   # materializacja od razu do Polars, bez przejścia przez Pandas

miasto,liczba_produktow
str,i64
"""Gdansk""",4
"""Krakow""",4
"""Poznan""",4
"""Warszawa""",4


> **Kiedy to się opłaca względem `sprzedaz.groupby(...)` w Pandas albo
> `sprzedaz_pl.group_by(...)` w Polars?** Przy prostych, pojedynczych
> agregacjach na danych mieszczących się wygodnie w pamięci różnice bywają
> marginalne — silnik Polars też jest wektoryzowany. DuckDB zaczyna wyraźnie
> wygrywać przy: (1) **złożonych zapytaniach z wieloma joinami/CTE**, gdzie
> optymalizator SQL układa plan lepiej niż ręczny łańcuch metod, (2) **danych
> większych niż RAM** (sekcja 5) oraz (3) gdy **łączysz kilka źródeł naraz**
> (sekcja 4) — tego Pandas/Polars same z siebie nie robią.

## 4. Joiny między różnymi źródłami w jednym zapytaniu

Mocna strona DuckDB jako "kleju" w pipeline: w jednym zapytaniu SQL możesz
połączyć DataFrame Pandas, DataFrame Polars i natywną tabelę DuckDB —
bez ręcznego ujednolicania typów czy konwersji przed joinem.

In [8]:
slownik_miast = pl.DataFrame({
    "miasto": ["Warszawa", "Krakow", "Poznan", "Gdansk"],
    "wojewodztwo": ["mazowieckie", "malopolskie", "wielkopolskie", "pomorskie"],
})

duckdb.sql("""
    SELECT sw.wojewodztwo,
           ROUND(SUM(s.ilosc * s.cena), 2) AS przychod
    FROM sprzedaz AS s                 -- Pandas
    JOIN slownik_miast AS sw           -- Polars
      ON s.miasto = sw.miasto
    GROUP BY sw.wojewodztwo
    ORDER BY przychod DESC
""").df()

,wojewodztwo,przychod
0,mazowieckie,63907713.36
1,pomorskie,63522819.80
2,wielkopolskie,63500762.68
3,malopolskie,63197594.03


## 5. Odczyt plików bezpośrednio — CSV/Parquet, bez ładowania do pamięci

`read_parquet()`/`read_csv()` w DuckDB nie ładują całego pliku do RAM przed
przetworzeniem — silnik czyta kolumnowo i strumieniowo, przepuszczając przez
zapytanie tylko potrzebne kolumny/wiersze (predicate/projection pushdown).
To kluczowa przewaga nad `pandas.read_parquet(...)` + filtrowanie w Pandas
przy plikach **większych niż dostępny RAM** — DuckDB potrafi je przetworzyć,
Pandas w typowym użyciu nie (chyba że ręcznie chunkujesz).

In [9]:
dane_dir = Path("dane_demo")
dane_dir.mkdir(exist_ok=True)

# Zapis partycjonowany wg miesiąca (hive-style) — symulacja typowego układu
# plików w data lake / na S3
duckdb.sql("SELECT * FROM sprzedaz").write_parquet(
    str(dane_dir / "sprzedaz.parquet"),
)
sprzedaz.to_csv(dane_dir / "sprzedaz.csv", index=False)

list(dane_dir.iterdir())

[PosixPath('dane_demo/sprzedaz.csv'), PosixPath('dane_demo/sprzedaz.parquet')]

In [10]:
# Parquet — schemat i typy kolumn czytane z metadanych pliku, bez zgadywania
duckdb.sql(f"SELECT * FROM read_parquet('{dane_dir}/sprzedaz.parquet') LIMIT 5").df()

,id_transakcji,produkt,miasto,ilosc,cena,data
0,1,A,Gdansk,7,244.87,2025-01-01 00:00:00
1,2,D,Gdansk,4,211.59,2025-01-01 00:01:00
2,3,C,Gdansk,9,385.66,2025-01-01 00:02:00
3,4,B,Gdansk,4,372.93,2025-01-01 00:03:00
4,5,B,Gdansk,4,376.92,2025-01-01 00:04:00


In [11]:
# CSV — z automatycznym wykrywaniem typów, separatora i nagłówka
# (przy nietypowym pliku warto override'ować: read_csv(..., sep=';', dtype={...}))
duckdb.sql(f"""
    SELECT produkt, COUNT(*) AS n
    FROM read_csv('{dane_dir}/sprzedaz.csv')
    GROUP BY produkt
""").df()

,produkt,n
0,A,49807
1,D,49922
2,C,50323
3,B,49948


### Wiele plików naraz — glob i skanowanie katalogów

`read_parquet()`/`read_csv()` przyjmują też **wzorzec glob** albo listę
ścieżek — DuckDB scala pliki w jedno logiczne źródło bez ręcznego
`pd.concat([pd.read_parquet(p) for p in pliki])`.

In [12]:
# Symulacja katalogu z wieloma plikami (np. codzienny eksport z pipeline'u)
for miesiac, czesc in sprzedaz.groupby(sprzedaz["data"].dt.month):
    czesc.to_parquet(dane_dir / f"sprzedaz_2025_{miesiac:02d}.parquet", index=False)

duckdb.sql(f"""
    SELECT COUNT(*) AS wierszy, COUNT(DISTINCT filename) AS liczba_plikow
    FROM read_parquet('{dane_dir}/sprzedaz_2025_*.parquet', filename = true)
""").df()

,wierszy,liczba_plikow
0,200000,5


## 6. Zapis wyników — `COPY`, `write_parquet`, powrót do DataFrame

Wynik zapytania możesz skierować dalej na trzy sposoby: zmaterializować do
DataFrame (jak dotąd), zapisać do pliku (`COPY ... TO` albo
`relacja.write_parquet(...)`), albo utworzyć nową tabelę DuckDB
(`CREATE TABLE ... AS SELECT ...`).

In [13]:
# Wariant SQL — elastyczny co do formatu (PARQUET/CSV/JSON) i opcji (kompresja)
duckdb.sql(f"""
    COPY (
        SELECT produkt, miasto, SUM(ilosc * cena) AS przychod
        FROM sprzedaz
        GROUP BY produkt, miasto
    ) TO '{dane_dir}/podsumowanie.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# Wariant obiektowy — skrót dla relacji (bez pisania COPY)
duckdb.sql("SELECT * FROM sprzedaz WHERE produkt = 'A'").write_parquet(
    str(dane_dir / "produkt_a.parquet")
)

sorted(p.name for p in dane_dir.glob("*.parquet"))

['podsumowanie.parquet',
 'produkt_a.parquet',
 'sprzedaz.parquet',
 'sprzedaz_2025_01.parquet',
 'sprzedaz_2025_02.parquet',
 'sprzedaz_2025_03.parquet',
 'sprzedaz_2025_04.parquet',
 'sprzedaz_2025_05.parquet']

## 7. Dialekt SQL DuckDB — co jest inne (i wygodniejsze) niż w T-SQL

DuckDB jest bliżej składni Postgresa/Snowflake/BigQuery niż T-SQL. Kilka
rzeczy, które realnie skracają zapytania względem tego, do czego przyzwyczaja
SQL Server:

- **`GROUP BY ALL`** — grupuje po wszystkich kolumnach z `SELECT`, które nie
  są agregacją; nie trzeba wypisywać/numerować kolumn drugi raz.
- **`SELECT * EXCLUDE (...)`** / **`SELECT * REPLACE (... AS kolumna)`** —
  wybierz wszystko poza jedną kolumną / podmień jedną kolumnę w locie, bez
  wypisywania reszty z osobna.
- **`QUALIFY`** — filtruje po funkcji okna (np. `ROW_NUMBER() OVER (...)`)
  bez owijania zapytania w CTE/subquery tylko po to, żeby dodać `WHERE`.
- **natywny `PIVOT`/`UNPIVOT`** — bez ręcznego `CASE WHEN` jak w T-SQL.
- **typy złożone `LIST`/`STRUCT`/`MAP`** — kolumny zagnieżdżone, przydatne
  np. przy pracy z danymi z API/JSON bez rozbijania na osobne tabele.

In [14]:
# GROUP BY ALL — bez wypisywania "produkt, miasto" drugi raz
duckdb.sql("""
    SELECT produkt, miasto, ROUND(SUM(ilosc * cena), 2) AS przychod
    FROM sprzedaz
    GROUP BY ALL
    ORDER BY przychod DESC
    LIMIT 5
""").df()

,produkt,miasto,przychod
0,C,Gdansk,16341490.37
1,A,Warszawa,16241317.35
2,D,Poznan,16159288.07
3,B,Warszawa,16150960.32
4,C,Krakow,15953596.82


In [15]:
# QUALIFY — top transakcja wg przychodu w każdym mieście, bez CTE
duckdb.sql("""
    SELECT miasto, produkt, ilosc, cena,
           ROW_NUMBER() OVER (PARTITION BY miasto ORDER BY ilosc * cena DESC) AS ranga
    FROM sprzedaz
    QUALIFY ranga = 1
""").df()

,miasto,produkt,ilosc,cena,ranga
0,Gdansk,B,9,499.92,1
1,Poznan,A,9,499.84,1
2,Krakow,B,9,499.98,1
3,Warszawa,B,9,499.97,1


In [16]:
# PIVOT — przychód wg produktu w kolumnach, bez CASE WHEN
duckdb.sql("""
    PIVOT sprzedaz
    ON produkt
    USING ROUND(SUM(ilosc * cena), 0)
    GROUP BY miasto
""").df()

,miasto,A,B,C,D
0,Gdansk,15499738.0,15932700.0,16341490.0,15748891.0
1,Poznan,15751045.0,15724838.0,15865592.0,16159288.0
2,Krakow,15544055.0,15880463.0,15953597.0,15819479.0
3,Warszawa,16241317.0,16150960.0,15626905.0,15888531.0


## 8. Wydajność, `EXPLAIN` i praca z danymi większymi niż RAM

- **`EXPLAIN`** pokazuje plan zapytania (bez wykonania), **`EXPLAIN
  ANALYZE`** wykonuje zapytanie i dokłada realny czas/liczbę wierszy na
  każdym etapie planu — pierwsze miejsce do sprawdzenia, gdy zapytanie jest
  wolniejsze niż się spodziewasz (analogicznie do planu wykonania w SSMS,
  ale czytelniejsze przy operacjach kolumnowych).
- **`SET memory_limit`** / **`SET threads`** — kontrola zasobów. Przy
  przekroczeniu limitu pamięci DuckDB **spilluje na dysk** (do
  `temp_directory`) zamiast się wywalać — to właśnie mechanizm, który
  pozwala przetwarzać dane większe niż RAM.
- Domyślnie DuckDB używa tylu wątków, ile rdzeni ma maszyna — ograniczanie
  ma sens głównie na współdzielonym środowisku (np. kontener z limitem CPU).

In [17]:
duckdb.sql("""
    EXPLAIN
    SELECT produkt, SUM(ilosc * cena) AS przychod
    FROM sprzedaz
    GROUP BY produkt
""").show()

┌───────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  explain_key  │                                                               

In [18]:
duckdb.sql("SET memory_limit = '2GB'")
duckdb.sql("SET threads = 4")
duckdb.sql("SET temp_directory = 'duckdb_temp'")   # katalog na spill przy dużych zapytaniach

duckdb.sql("SELECT current_setting('memory_limit') AS memory_limit, "
            "current_setting('threads') AS threads").df()

,memory_limit,threads
0,1.8 GiB,4


## 9. Rozszerzenia (extensions)

DuckDB ma minimalny rdzeń, a funkcjonalność dokłada się rozszerzeniami
(`INSTALL`/`LOAD`, jednorazowo pobierane z internetu, potem działają
offline). Dwa najbardziej przydatne w Twoim stacku:

- **`spatial`** — funkcje `ST_*` (jak w PostGIS), odczyt/zapis Shapefile,
  GeoPackage, **GeoParquet**. Uzupełnia GeoPandas: duże, kolumnowe
  operacje przestrzenne (filtrowanie, joiny przestrzenne na milionach
  geometrii) często szybciej zrobisz w DuckDB SQL, a wynik wczytasz do
  GeoPandas/QGIS tylko do wizualizacji i dalszej edycji.
- **`httpfs`** — odczyt Parquet/CSV bezpośrednio z `s3://`, `https://`,
  Azure Blob — bez pobierania pliku na dysk najpierw.

Poniższe komórki wymagają dostępu do internetu (pobranie rozszerzenia przy
pierwszym użyciu) — kod referencyjny, nie wykonywany w tym notebooku.

In [19]:
# --- spatial: przykład joina przestrzennego bez GeoPandas w pętli ---
# duckdb.sql("INSTALL spatial; LOAD spatial;")
# duckdb.sql("""
#     SELECT p.id_punktu, r.nazwa_rejonu
#     FROM read_parquet('punkty.parquet') AS p
#     JOIN ST_Read('rejony.gpkg') AS r
#       ON ST_Within(ST_Point(p.lon, p.lat), r.geom)
# """).df()

# --- httpfs: odczyt Parquet z S3 bez pobierania na dysk ---
# duckdb.sql("INSTALL httpfs; LOAD httpfs;")
# duckdb.sql("SET s3_region = 'eu-central-1';")
# duckdb.sql("SELECT * FROM read_parquet('s3://moj-bucket/dane/*.parquet') LIMIT 10").df()

## 10. DuckDB jako silnik transformacji w pipeline'ie ETL/ELT z SQL Server

DuckDB nie ma natywnego łącznika do SQL Server (nie ma czego `ATTACH`) —
typowy wzorzec w Twoim stacku to trzy kroki, gdzie DuckDB odpowiada tylko
za środkowy, obliczeniowy etap:

1. **Ekstrakcja** — `mssql_python`/`pyodbc` → wynik zapytania jako
   Pandas/Polars DataFrame (patrz osobny notebook o `mssql_python`).
2. **Transformacja** — ciężkie agregacje/joiny/przekształcenia jako SQL w
   DuckDB, operując bezpośrednio na tych DataFrame'ach (sekcje 3–4 wyżej) —
   szczególnie opłacalne, gdy łączysz dane z SQL Servera z plikami
   Parquet/CSV z innych źródeł w jednym zapytaniu.
3. **Load** — wynik z powrotem do SQL Servera (`executemany`/wzorce z
   `mssql_python`) albo zapis do Parquet, jeśli docelowym odbiorcą jest
   Power BI (import z pliku/Data Lake) albo kolejny etap w Airflow.

Podział odpowiedzialności wart rozważenia: proste filtry/projekcje, które i
tak silnik SQL Server umie zoptymalizować (wykorzystanie indeksów), lepiej
zostawić w zapytaniu źródłowym po stronie SQL Servera — nie ma sensu
ściągać więcej danych niż trzeba tylko po to, żeby przefiltrować je
dopiero w DuckDB. DuckDB włączaj tam, gdzie **łączysz wiele źródeł** albo
wykonujesz operacje, których SQL Server po stronie serwera nie zrobi
wygodnie (albo obciążyłby produkcyjną bazę).

## 11. Czego DuckDB nie jest — ograniczenia

- **Jeden proces, jeden zapisujący na raz.** Plik `.duckdb` może być
  otwarty do zapisu przez jedno połączenie jednocześnie (wielu czytelników
  w trybie tylko-odczyt jest już wspierane, ale to wciąż nie jest
  wielodostępowy serwer bazodanowy). Nie zastępuje SQL Servera jako
  współdzielonego backendu dla wielu aplikacji/użytkowników naraz.
- **Nie jest bazą transakcyjną (OLTP).** Świetny do analityki wsadowej
  (agregacje, transformacje, raportowanie), słaby wybór pod częste,
  pojedyncze `INSERT`/`UPDATE` z wielu równoległych źródeł.
- **Skaluje się w górę (jedna maszyna), nie na zewnątrz (klaster).** Przy
  danych, które przestają mieścić się nawet z mechanizmem spill-to-disk na
  jednej maszynie, naturalnym krokiem jest już coś typu BigQuery/Spark —
  DuckDB dobrze sprawdza się jako narzędzie do prototypowania takich
  zapytań lokalnie, zanim trafią np. do BigQuery.

Najlepsze zastosowanie w Twoim stacku: **lokalny, szybki silnik
analityczny/ETL** — łączenie plików i DataFrame'ów, cięższe transformacje
przed zapisem do SQL Servera albo przed wizualizacją w Power BI — nie
zamiennik samego SQL Servera.